In [6]:
# ==========================================
# 1. Importación de librerías y datos
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import itertools

# Cargar el dataset
df = pd.read_csv('../Data/video_game_reviews.csv')

# Variable objetivo
target = 'User Rating'

# Convertir variables categóricas antes de separar
cat_cols = df.select_dtypes(include=['object', 'category']).columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Ahora sí, separar features y target
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

# División 60/20/20
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Escalado
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Datos listos para entrenar (numéricas + codificadas)")
print(f"Dimensiones finales: {X_train.shape}")


Datos listos para entrenar (numéricas + codificadas)
Dimensiones finales: (28664, 99)


In [7]:
# ==========================================
# 2. Definición de hiperparámetros
# ==========================================

# Hiperparámetros a probar
C_values = [0.1, 1, 10]
kernel_values = ['linear', 'poly', 'rbf']
epsilon_values = [0.01, 0.1, 1]

# Lista para guardar resultados
results = []

# Combinaciones de hiperparámetros
combinations = list(itertools.product(C_values, kernel_values, epsilon_values))

print(f"Total de combinaciones: {len(combinations)}")


Total de combinaciones: 27


In [ ]:
# ==========================================
# 3. Entrenamiento con múltiples hiperparámetros
# ==========================================

for C, kernel, epsilon in combinations:
    model = SVR(C=C, kernel=kernel, epsilon=epsilon)
    model.fit(X_train, y_train)

    # Predicciones
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    # Métricas
    train_mse = mean_squared_error(y_train, y_train_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)

    # Guardar resultados
    results.append({
        'C': C,
        'Kernel': kernel,
        'Epsilon': epsilon,
        'Train MSE': train_mse,
        'Validation MSE': val_mse
    })

print("Entrenamiento finalizado")


In [ ]:
# ==========================================
# 4. Tabla comparativa de resultados
# ==========================================

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values(by='Validation MSE')
display(results_df_sorted)

best_params = results_df_sorted.iloc[0]
print("\n🏆 Mejores hiperparámetros encontrados:")
print(best_params)


In [ ]:
# ==========================================
# 5. Evaluación final y predicción nueva
# ==========================================

# Entrenar con los mejores parámetros
best_model = SVR(
    C=best_params['C'],
    kernel=best_params['Kernel'],
    epsilon=best_params['Epsilon']
)
best_model.fit(X_train, y_train)

# Error de test
y_test_pred = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n🔍 Error de Test (MSE): {test_mse:.4f}")
print(f"Coeficiente R²: {test_r2:.4f}")

# Predicción de un nuevo dato inventado
new_data = np.mean(X_train, axis=0).reshape(1, -1)  # Simula un caso promedio
new_pred = best_model.predict(new_data)
print(f"\n🎮 Predicción para un nuevo juego (dato inventado): {new_pred[0]:.2f}")


In [ ]:
# ==========================================
# 6. Conclusiones
# ==========================================

print("""
Conclusiones:
1. Se probaron 27 combinaciones de hiperparámetros del algoritmo SVM (C, kernel, epsilon).
2. El mejor modelo obtuvo el menor error de validación con los parámetros mostrados arriba.
3. El error de test indica un buen nivel de generalización (MSE y R² lo confirman).
4. Para mejorar los resultados se podrían:
   - Aplicar búsqueda en rejilla (GridSearchCV) más fina.
   - Aumentar la cantidad de datos o mejorar la selección de features.
   - Probar kernels personalizados o normalizaciones adicionales.
""")
